In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Input e output delle primitive

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** La versione beta di un nuovo modello di esecuzione è ora disponibile. Il modello di esecuzione diretto offre maggiore flessibilità nella personalizzazione del flusso di lavoro per la mitigazione degli errori. Consulta la guida al [modello di esecuzione diretto](/guides/directed-execution-model) per ulteriori informazioni.


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.4.0
```
</details>
Questa pagina offre una panoramica degli input e degli output delle primitive Qiskit Runtime che eseguono workload sulle risorse di calcolo di IBM Quantum&reg;. Queste primitive ti permettono di definire in modo efficiente workload vettorizzati tramite una struttura dati nota come **Primitive Unified Bloc (PUB)**. I PUB sono l'unità fondamentale di lavoro che una QPU deve eseguire per portare a termine questi workload. Vengono usati come input al metodo [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) per le primitive Sampler ed Estimator, che eseguono il workload definito come job. Una volta completato il job, i risultati vengono restituiti in un formato che dipende sia dai PUB usati sia dalle opzioni di runtime specificate dalla primitiva Sampler o Estimator.
<span id="pubs"></span>
## Panoramica dei PUB
Quando si richiama il metodo [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) di una primitiva, l'argomento principale richiesto è una `list` di una o più tuple — una per ogni circuito eseguito dalla primitiva. Ognuna di queste tuple è considerata un PUB, e gli elementi richiesti in ciascuna tupla dipendono dalla primitiva usata. I dati forniti a queste tuple possono anche essere disposti in una varietà di forme per offrire flessibilità nel workload tramite broadcasting — le cui regole sono descritte in una [sezione successiva](#broadcasting-rules).

### PUB dell'Estimator
Per la primitiva Estimator, il formato del PUB può contenere al massimo quattro valori:
- Un singolo `QuantumCircuit`, che può contenere uno o più oggetti [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Una lista di uno o più osservabili, che specificano i valori di aspettazione da stimare, organizzati in un array (per esempio, un singolo osservabile rappresentato come array 0-d, una lista di osservabili come array 1-d, ecc.). I dati possono essere in uno qualsiasi dei formati `ObservablesArrayLike`, come `Pauli`, `SparsePauliOp`, `PauliList` o `str`.
   > **Note:** Se hai due osservabili che commutano in PUB diversi ma con lo stesso circuito, non verranno stimati usando la stessa misurazione. Ogni PUB rappresenta una base di misurazione diversa, quindi sono necessarie misurazioni separate per ciascun PUB. Per assicurarti che gli osservabili commutanti vengano stimati con la stessa misurazione, devono essere raggruppati nello stesso PUB.
- Una raccolta di valori parametrici da collegare al circuito. Può essere specificata come un singolo oggetto array-like in cui l'ultimo indice è relativo agli oggetti `Parameter` del circuito, oppure omessa (o equivalentemente impostata a `None`) se il circuito non contiene oggetti `Parameter`.
- (Facoltativamente) una precisione target per i valori di aspettazione da stimare

### PUB del Sampler
Per la primitiva Sampler, il formato della tupla PUB contiene al massimo tre valori:
- Un singolo `QuantumCircuit`, che può contenere uno o più oggetti [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
   *Nota: questi circuito devono includere anche istruzioni di misurazione per ciascuno dei qubit da campionare.*
- Una raccolta di valori parametrici da collegare al circuito $\theta_k$ (necessaria solo se vengono usati oggetti `Parameter` che devono essere associati a runtime)
- (Facoltativamente) un numero di shot con cui misurare il circuito
---

Il codice seguente mostra un esempio di input vettorizzati per la primitiva `Estimator` ed esegue questi input su un backend IBM&reg; come un singolo oggetto `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

<span id="broadcasting"></span>
### Broadcasting rules

The PUBs aggregate elements from multiple arrays (observables and parameter values) by following the same broadcasting rules as NumPy. This section briefly summarizes those rules.  For a detailed explanation, see the [NumPy broadcasting rules documentation](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Rules:

* Input arrays do not need to have the same number of dimensions.
  * The resulting array will have the same number of dimensions as the input array with the largest dimension.
  * The size of each dimension is the largest size of the corresponding dimension.
  * Missing dimensions are assumed to have size one.
* Shape comparisons start with the rightmost dimension and continue to the left.
* Two dimensions are compatible if their sizes are equal or if one of them is 1.

Examples of array pairs that broadcast:

```text
A1     (1d array):      1
A2     (2d array):  3 x 5
Result (2d array):  3 x 5


A1     (3d array):  11 x 2 x 7
A2     (3d array):  11 x 1 x 7
Result (3d array):  11 x 2 x 7
```

Examples of array pairs that do not broadcast:

```text
A1     (1d array):  5
A2     (1d array):  3

A1     (2d array):      2 x 1
A2     (3d array):  6 x 5 x 4 # This would work if the middle dimension were 2, but it is 5.
```

`Estimator` returns one expectation value estimate for each element of the broadcasted shape.

Here are some examples of common patterns expressed in terms of array broadcasting.  Their accompanying visual representation is shown in the figure that follows:


Parameter value sets are represented by n x m arrays, and observable arrays are represented by one or more single-column arrays. For each example in the previous code, the parameter value sets are combined with their observable array to create the resulting expectation value estimates.

 - *Example 1*: (broadcast single observable) has a parameter value set that is a 5x1 array and a 1x1 observables array.  The one item in the observables array is combined with each item in the parameter value set to create a single 5x1 array where each item is a combination of the original item in the parameter value set with the item in the observables array.

 - *Example 2*: (zip) has a 5x1 parameter value set and a 5x1 observables array.  The output is a 5x1 array where each item is a combination of the nth item in the parameter value set with the nth item in the observables array.

  - *Example 3*: (outer/product) has a 1x6 parameter value set and a 4x1 observables array.  Their combination results in a 4x6 array that is created by combining each item in the parameter value set with *every* item in the observables array, and thus each parameter value becomes an entire column in the output.

  - *Example 4*: (Standard nd generalization) has a 3x6 parameter value set array and two 3x1 observables array.  These combine to create two 3x6 output arrays in a similar manner to the previous example.

![This image illustrates several visual representations of array broadcasting.](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual representation of broadcasting")

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

### Regole di broadcasting
I PUB aggregano elementi da più array (osservabili e valori parametrici) seguendo le stesse regole di broadcasting di NumPy. Questa sezione riassume brevemente tali regole. Per una spiegazione dettagliata, consulta la [documentazione sulle regole di broadcasting di NumPy](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Regole:

* Gli array di input non devono avere necessariamente lo stesso numero di dimensioni.
  * L'array risultante avrà lo stesso numero di dimensioni dell'array di input con la dimensione maggiore.
  * La dimensione di ciascuna dimensione è la dimensione più grande della dimensione corrispondente.
  * Le dimensioni mancanti si assumono di dimensione uno.
* I confronti tra forme partono dalla dimensione più a destra e proseguono verso sinistra.
* Due dimensioni sono compatibili se le loro dimensioni sono uguali o se una di esse è 1.

Esempi di coppie di array che fanno broadcasting:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

Esempi di coppie di array che non fanno broadcasting:

In [4]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  3 255]
 [  3 255]]



`EstimatorV2` restituisce una stima del valore di aspettazione per ciascun elemento della forma trasmessa (broadcasted).

Di seguito sono riportati alcuni esempi di pattern comuni espressi in termini di broadcasting di array. La loro rappresentazione visiva è mostrata nella figura seguente:

I set di valori parametrici sono rappresentati da array n x m, e gli array di osservabili sono rappresentati da uno o più array a colonna singola. Per ciascun esempio nel codice precedente, i set di valori parametrici vengono combinati con il loro array di osservabili per creare le stime dei valori di aspettazione risultanti.

 - *Esempio 1*: (broadcast di un singolo osservabile) ha un set di valori parametrici che è un array 5x1 e un array di osservabili 1x1. Il singolo elemento nell'array di osservabili viene combinato con ciascun elemento del set di valori parametrici per creare un singolo array 5x1 in cui ogni elemento è una combinazione dell'elemento originale nel set di valori parametrici con l'elemento nell'array di osservabili.

 - *Esempio 2*: (zip) ha un set di valori parametrici 5x1 e un array di osservabili 5x1. L'output è un array 5x1 in cui ogni elemento è una combinazione dell'n-esimo elemento del set di valori parametrici con l'n-esimo elemento dell'array di osservabili.

  - *Esempio 3*: (outer/product) ha un set di valori parametrici 1x6 e un array di osservabili 4x1. La loro combinazione produce un array 4x6 creato combinando ciascun elemento del set di valori parametrici con *ogni* elemento dell'array di osservabili; quindi ogni valore parametrico diventa un'intera colonna nell'output.

  - *Esempio 4*: (generalizzazione nd standard) ha un array di set di valori parametrici 3x6 e due array di osservabili 3x1. Questi si combinano per creare due array di output 3x6 in modo simile all'esempio precedente.

![Questa immagine illustra diverse rappresentazioni visive del broadcasting di array](../docs/images/guides/primitive-input-output/broadcasting.svg "Rappresentazione visiva del broadcasting")

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 492, '1111111111': 532}


<Admonition type="tip" title="SparsePauliOp">
Ogni `SparsePauliOp` conta come un singolo elemento in questo contesto, indipendentemente dal numero di Pauli contenuti nello `SparsePauliOp`. Pertanto, ai fini di queste regole di broadcasting, tutti i seguenti elementi hanno la stessa forma:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


Le seguenti liste di operatori, pur essendo equivalenti in termini di informazioni contenute, hanno forme diverse:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
</Admonition>
## Panoramica degli output delle primitive
Una volta che uno o più PUB vengono inviati a una QPU per l'esecuzione e un job viene completato con successo, i dati vengono restituiti come oggetto contenitore [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) a cui si accede chiamando il metodo `RuntimeJobV2.result()`. Il `PrimitiveResult` contiene una lista iterabile di oggetti [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) che contengono i risultati di esecuzione per ciascun PUB. A seconda della primitiva usata, questi dati saranno o valori di aspettazione con le relative barre di errore (nel caso dell'Estimator), o campioni dell'output del circuito (nel caso del Sampler).

Ogni elemento di questa lista corrisponde a ciascun PUB inviato al metodo `run()` della primitiva (per esempio, un job inviato con 20 PUB restituirà un oggetto `PrimitiveResult` che contiene una lista di 20 `PubResult`, uno per ogni PUB).

Ognuno di questi oggetti `PubResult` possiede sia un attributo `data` sia un attributo `metadata`. L'attributo `data` è un [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personalizzato che contiene i valori di misurazione effettivi, le deviazioni standard e così via. Questo `DataBin` ha vari attributi a seconda della forma o struttura del PUB associato e delle opzioni di mitigazione degli errori specificate dalla primitiva usata per inviare il job (per esempio, [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) o [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)). L'attributo `metadata`, invece, contiene informazioni sulle opzioni di runtime e di mitigazione degli errori usate (spiegate più avanti nella sezione [Metadati dei risultati](#result-metadata) di questa pagina).

Di seguito è riportato uno schema visivo della struttura dati `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Estimator Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data for first PUB (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second PUB
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Sampler Output">

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 255]
 [  1 255]
 ...
 [  0   0]
 [  0   0]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 255]
 [  1 255]
 [  1 255]
 [  1 255]]



Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: -0.017578125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: -0.017578125

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  0   0]
 [  0   0]
 [  3 255]]



</TabItem>
</Tabs>

In sintesi, un singolo job restituisce un oggetto `PrimitiveResult` che contiene una lista di uno o più oggetti `PubResult`. Questi oggetti `PubResult` memorizzano poi i dati di misurazione per ciascun PUB inviato al job.

Ogni `PubResult` ha formati e attributi diversi a seconda del tipo di primitiva usata per il job. I dettagli sono spiegati di seguito.

### Output dell'Estimator
Ogni `PubResult` per la primitiva Estimator contiene almeno un array di valori di aspettazione (`PubResult.data.evs`) e le deviazioni standard associate (o `PubResult.data.stds` o `PubResult.data.ensemble_standard_error` a seconda del `resilience_level` usato), ma può contenere ulteriori dati a seconda delle opzioni di mitigazione degli errori specificate.

Il frammento di codice seguente descrive il formato del `PrimitiveResult` (e del `PubResult` associato) per il job creato in precedenza.

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},


## Next steps

<Admonition type="tip" title="Recommendations">

  -  Review the [Qiskit primitives](/docs/api/qiskit/primitives) API.
  -  Review the [Qiskit Aer primitives](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) API.
  -  Learn more about the [Qiskit Runtime primitives](/docs/guides/qiskit-runtime-primitives).
  -  Review the [Qiskit Runtime Estimator](/docs/api/qiskit-ibm-runtime/estimator-v2) API.
  -  Review the [Qiskit Runtime Sampler](/docs/api/qiskit-ibm-runtime/sampler-v2) API.
</Admonition>